### 01-tool-use-from-scratch

In [ ]:
import requests, json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1"


def get_temperature(city: str) -> float:
    print("Fetching temperature for:", city)
    return 20.0


def main():
    user_input = input("Your question: ")

    prompt = f"""
                You are a helpful assistant. Answer the user's question in a friendly way.

                You can also use tools if you feel like they help you provide a better answer:
                - get_temperature(city: str) -> float: Get the current temperature for a given city.

                If you want to use one of these tools, you should output the tool name and its arguments in the following format:
                tool_name: arg1

                For example:
                get_temperature: New York

                With that in mind, answer the user's question:

                <user-question>
                {user_input}
                </user-question>

                If you request a tool, please output ONLY the tool call and nothing else.
            """

    payload = {
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False
                }

    response = requests.post(OLLAMA_URL, json=payload)
    response.raise_for_status()

    reply = response.json()["response"]

    print("Model reply:", reply)

    # TOOL CALL
    if reply.startswith("get_temperature:"):
        arg = reply.split(":", 1)[1].strip()

        temperature = get_temperature(arg)

        second_prompt = f"""
                            You are a helpful assistant. Answer the user's question in a friendly way.

                            User question:
                            {user_input}

                            Tool result:
                            The current temperature in {arg} is {temperature}°C.
                        """

        payload = {
                    "model": MODEL_NAME,
                    "prompt": second_prompt,
                    "stream": False
                    }

        response = requests.post(OLLAMA_URL, json=payload)
        response.raise_for_status()

        final_reply = response.json()["response"]

        print(final_reply)

    else:
        print(reply)


if __name__ == "__main__":
    main()

### 02-ollama-functions

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "deepseek-r1"


def get_temperature(city: str) -> float:
    print("Fetching temperature for:", city)
    return 20.0


def call_ollama(prompt: str, system: str | None = None, response_format=None) -> str:
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
    }

    if system:
        payload["system"] = system

    if response_format is not None:
        payload["format"] = response_format

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["response"]


def main():
    user_input = input("Your question: ").strip()

    decision_schema = {
        "type": "object",
        "properties": {
            "action": {
                "type": "string",
                "enum": ["tool", "answer"]
            },
            "tool_name": {
                "type": "string"
            },
            "arguments": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string"
                    }
                },
                "additionalProperties": False
            },
            "answer": {
                "type": "string"
            }
        },
        "required": ["action"],
        "additionalProperties": False
    }

    system_prompt = (
                    "You are a helpful assistant. "
                    "You can answer normally, or decide to use a tool when needed. "
                    "Available tool: get_temperature(city: str) -> float. "
                    "If the user asks about current weather or temperature for a city, prefer the tool."
                    )

    first_prompt = f"""
                        Return ONLY valid JSON matching the provided schema.

                        Available tool:
                        - get_temperature(city: str) -> float

                        User question:
                        {user_input}
                    """.strip()

    raw_reply = call_ollama(prompt=first_prompt, system=system_prompt, response_format=decision_schema)

    try:
        decision = json.loads(raw_reply)
    except json.JSONDecodeError:
        print("Model returned invalid JSON:")
        print(raw_reply)
        return

    action = decision.get("action")

    if action == "tool":
        tool_name = decision.get("tool_name")
        arguments = decision.get("arguments", {})

        if tool_name != "get_temperature":
            print(f"Unknown tool requested by model: {tool_name}")
            return

        city = arguments.get("city")
        if not city:
            print("Tool call is missing required argument: city")
            return

        temperature = get_temperature(city)

        second_prompt = f"""
                            The user asked:
                            {user_input}

                            Tool used:
                            get_temperature(city="{city}")

                            Tool result:
                            The current temperature in {city} is {temperature}°C.

                            Now answer the user in a friendly way.
                        """.strip()

        final_reply = call_ollama(prompt=second_prompt, system="You are a helpful assistant. Answer clearly and briefly.")

        print(final_reply)

    elif action == "answer":
        print(decision.get("answer", "No answer returned by model."))

    else:
        print("Unexpected action from model:")
        print(decision)

if __name__ == "__main__":
    main()